In [1]:
!pip3 install pandas
import pandas as pd
import numpy as np
import json

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.2.3-cp39-cp39-macosx_11_0_arm64.whl (11.3 MB)
  Using cached tzdata-2024.2-py2.py3-none-any.whl (346 kB)
  Using cached pytz-2024.2-py2.py3-none-any.whl (508 kB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Define the metadata

In [4]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://doi.org/10.1038/s41586-022-04518-2"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-022-04518-2/MediaObjects/41586_2022_4518_MOESM4_ESM.xlsx"

# don't include in header
table_name = "41586_2022_4518_MOESM4_ESM.xlsx"
table_name = "clean_degs.xlsx" # local name

## Read in file

In [5]:
!pip3 install openpyxl

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 250 kB 4.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
excel = pd.read_excel(table_name, skiprows = 1)

df = excel.rename(columns={"cluster": "celltype"})

In [7]:
df.head()

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,celltype,gene
0,CXCL14,0.0,3.699825,0.426,0.082,0.0,ASPC,CXCL14
1,NEGR1,0.0,3.668332,0.867,0.247,0.0,ASPC,NEGR1
2,DCN,0.0,3.565394,0.960,0.506,0.0,ASPC,DCN
3,LAMA2,0.0,3.417902,0.829,0.266,0.0,ASPC,LAMA2
4,APOD,0.0,3.383653,0.493,0.120,0.0,ASPC,APOD


## Perform the filtering

In [8]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [9]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [10]:
markers

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,celltype,gene
8248,S100A9,0.0,2.272305,0.239,0.045,0.0,monocyte,S100A9
11080,CXCL81,0.0,2.446044,0.243,0.013,0.0,dendritic_cell,CXCL8
15121,KCNQ54,0.0,2.205825,0.282,0.032,0.0,nk_cell,KCNQ5
15101,CCL4,0.0,2.948546,0.317,0.016,0.0,nk_cell,CCL4
11069,AREG1,0.0,3.337181,0.345,0.018,0.0,dendritic_cell,AREG
...,...,...,...,...,...,...,...,...
2,DCN,0.0,3.565394,0.960,0.506,0.0,ASPC,DCN
3035,SORBS1,0.0,3.347645,0.967,0.170,0.0,adipocyte,SORBS1
3033,PDE3B,0.0,3.599566,0.974,0.218,0.0,adipocyte,PDE3B
3045,GHR,0.0,2.701779,0.977,0.417,0.0,adipocyte,GHR


In [11]:
markers[col_ct].value_counts()

celltype
pericyte          20
SMC               20
b_cell            20
neutrophil        20
endometrium       20
ASPC              20
endothelial       20
mast_cell         20
LEC               20
macrophage        20
mesothelium       20
adipocyte         20
dendritic_cell    19
nk_cell           17
t_cell            15
monocyte          12
Name: count, dtype: int64

In [12]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

celltype
nk_cell           0.509353
t_cell            0.533600
monocyte          0.560417
pericyte          0.617000
neutrophil        0.627000
SMC               0.637950
b_cell            0.647900
dendritic_cell    0.661579
endometrium       0.679300
LEC               0.748200
endothelial       0.748200
ASPC              0.753150
mast_cell         0.753800
macrophage        0.770350
mesothelium       0.811800
adipocyte         0.926800
Name: pct.1, dtype: float64

## Convert to evidence json

In [13]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
save = df.to_dict(orient = "records")

In [14]:
save

[{'cell_type_label': 'monocyte',
  'gene': 'S100A9',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'dendritic_cell',
  'gene': 'CXCL8',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'nk_cell',
  'gene': 'KCNQ5',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'nk_cell',
  'gene': 'CCL4',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'dendritic_cell',
  'gene': 'AREG',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'dendritic_cell',
  'gene': 'FCER1A',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 't_cell',
  'gene': 'LINC01934',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'monocyte',
  'gene': 'RILPL2'

## Save evidence

In [15]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 